# CP4 · Examen blanc de la partie IV — vérification de ta copie

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cemah2/workbookMaths/blob/main/checkpoints/partie_4/02_examen_notebook.ipynb)

L'examen se fait **sur papier** (`01_examen_sujet.md`, 110 minutes, fiches fermées). Ce notebook ne sert qu'**après** : il vérifie les réponses chiffrées de ta copie. Ses cellules ne vérifient rien tant que `EXAM_OVER` vaut `False`, et « Tout exécuter » (*Run all*) va jusqu'au bout même si rien n'est rempli (⏸️).

> Travaille dans **ta copie** (`mon_travail/checkpoints/partie_4/02_examen_notebook.ipynb`, créée par `python tools/start_chapter.py CP4`) : ce fichier-ci est mis à jour par Claude. Sur Colab, le badge ouvre la version du dépôt, qui n'est pas enregistrée : crée puis ouvre ta copie comme l'explique `00_setup/COLAB.md` §2.

In [ ]:
# ⚙️ SETUP: run this cell first (on Colab or on your computer)
FAST_MODE = True  # True = quick version (a few seconds per cell); False = bigger experiments
SEED = 42

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/cemah2/workbookMaths.git"
DRIVE_DIR = Path("/content/drive/MyDrive/workbookMaths")

try:
    import google.colab  # noqa: F401  (this module only exists on Colab)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    subprocess.run(["git", "config", "--global", "--add", "safe.directory", str(DRIVE_DIR)])
    if not (DRIVE_DIR / ".git").exists():
        subprocess.run(["git", "clone", REPO_URL, str(DRIVE_DIR)], check=True)
    else:
        # Get the new chapters. Your own work only lives in mon_travail/, which Claude never
        # touches, so "rebase + autostash" keeps it safe (even with local commits).
        identity = subprocess.run(["git", "-C", str(DRIVE_DIR), "config", "user.email"],
                                  capture_output=True).returncode == 0
        who = [] if identity else ["-c", "user.name=Workbook", "-c", "user.email=workbook@localhost"]
        pull = subprocess.run(["git", *who, "-C", str(DRIVE_DIR), "pull", "--rebase", "--autostash"],
                              capture_output=True, text=True)
        conflicts = subprocess.run(["git", "-C", str(DRIVE_DIR), "diff", "--name-only", "--diff-filter=U"],
                                   capture_output=True, text=True).stdout.split()
        if conflicts:  # a file changed outside mon_travail/ AND by Claude: git leaves conflict markers in it
            print("⚠️ Tes modifications de " + ", ".join(conflicts) + " (hors de mon_travail/) entrent en conflit "
                  "avec la nouvelle version ; elles sont gardées dans `git stash`. Pour reprendre la version du "
                  "dépôt : git restore --source=HEAD --staged --worktree <fichier> (00_setup/COLAB.md, « Problèmes "
                  "fréquents »).")
        elif pull.returncode != 0:
            print("⚠️ git pull a échoué (voir 00_setup/COLAB.md, « Problèmes fréquents ») :\n" + pull.stderr)
    ROOT = DRIVE_DIR
else:
    ROOT = Path(os.environ.get("WBM_ROOT", Path.cwd())).resolve()
    while not (ROOT / "src" / "wbm").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    if not (ROOT / "src" / "wbm").exists():
        raise FileNotFoundError("Dépôt introuvable : ouvre ce notebook depuis le dossier de workbookMaths.")

os.environ["WBM_ROOT"] = str(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import wbm  # noqa: E402

cfg = wbm.setup(seed=SEED, fast=FAST_MODE)
FAST_MODE = cfg.fast

## Après l'examen : vérification automatique

**Ne commence qu'une fois ta copie terminée**, examen rendu. Ce notebook vérifie les réponses que tu as écrites sur ta feuille, question par question : ✅ juste, ❌ faux (avec une piste), ⏳ pas rempli. Il ne note pas : les démarches, les justifications, le quiz CP4.Q1, la démonstration CP4.8 et les corrections de CP4.9 b) se corrigent avec `03_examen_corrige.md`, qui donne aussi le barème.

1. Passe `EXAM_OVER` à `True` dans la cellule ci-dessous, puis exécute-la.
2. Reporte chaque réponse de ta feuille : **la valeur** que tu as écrite, pas un nouveau calcul (sinon tu ne vérifies plus ta copie). Laisse `...` pour une question que tu n'as pas faite.
3. Si une cellule s'arrête sur une erreur, « Tout exécuter » s'arrête avec elle : corrige-la, puis relance cette cellule et les suivantes, jusqu'au bilan.
4. Les formes d'écriture : en Python, le séparateur décimal est un **point** (`2.75`) : une virgule arrête toute la cellule (`0,027`), et aucune réponse de la question n'est vérifiée, ou donne un couple de nombres (`0,75`) ; une liste s'écrit entre crochets (`[-1.5, 4]`), une matrice comme une liste de lignes (`[[1, -2], [0.5, 3]]`) ; un texte entre guillemets (`"XYZ"`), comme une fraction (`"3/8"`) ou une forme (`"(4, 5)"`) ; un numéro de choix s'écrit sans guillemets.
5. Le bilan, à la fin, dit aussi quelles réponses n'ont pas été vérifiées.

In [ ]:
EXAM_OVER = False   # set it to True once your exam is finished, then run the cells below

RESULTS = {}        # sub-ID -> result of wbm.check
_REMINDED = []      # the full reminder is printed once, then a short one


def exam_over():
    """True once the exam is finished; otherwise print a reminder (and the cell checks nothing)."""
    if not EXAM_OVER:
        print("⏸️ Rien n'est vérifié (EXAM_OVER vaut False)." if _REMINDED else
              "⏸️ Vérification après l'examen : passe EXAM_OVER à True dans cette cellule, puis exécute-la.")
        _REMINDED.append(True)
        return False
    if "wbm" not in globals():
        print("⚠️ Exécute d'abord la cellule de configuration, tout en haut du notebook.")
        return False
    return True

**CP4.1 — Coordonnées homogènes, chemins d'un graphe et variance d'un produit**

In [ ]:
# CP4.1: the values written on your answer sheet
answer_CP4_1a = ...  # a) a list of 3 lists
answer_CP4_1b = ...  # b) a list of 2 numbers
answer_CP4_1c = ...  # c) a list of 3 numbers
answer_CP4_1d = ...  # d) an integer
answer_CP4_1e = ...  # e) an integer
answer_CP4_1f = ...  # f) a number
answer_CP4_1g = ...  # g) a number
answer_CP4_1h = ...  # h) a number (4 decimals)

if exam_over():
    for letter, answer in zip("abcdefgh", [answer_CP4_1a, answer_CP4_1b, answer_CP4_1c, answer_CP4_1d, answer_CP4_1e, answer_CP4_1f, answer_CP4_1g, answer_CP4_1h]):
        RESULTS[f"CP4.1{letter}"] = wbm.check(f"CP4.1{letter}", answer)

**CP4.2 — Jacobienne d'une composée et produit vecteur-jacobienne**

In [ ]:
# CP4.2: the values written on your answer sheet
answer_CP4_2a = ...  # a) a list of 3 numbers
answer_CP4_2b = ...  # b) a list of 3 lists
answer_CP4_2c = ...  # c) a list of 2 lists
answer_CP4_2d = ...  # d) [[a, b], [c, d]]
answer_CP4_2e = ...  # e) a list of 2 numbers
answer_CP4_2f = ...  # f) a list of 2 integers
answer_CP4_2g = ...  # g) a list of 2 numbers
answer_CP4_2h = ...  # h) a list of 3 lists

if exam_over():
    for letter, answer in zip("abcdefgh", [answer_CP4_2a, answer_CP4_2b, answer_CP4_2c, answer_CP4_2d, answer_CP4_2e, answer_CP4_2f, answer_CP4_2g, answer_CP4_2h]):
        RESULTS[f"CP4.2{letter}"] = wbm.check(f"CP4.2{letter}", answer)

**CP4.3 — Mode direct et mode inverse sur un graphe de calcul**

In [ ]:
# CP4.3: the values written on your answer sheet
answer_CP4_3a = ...  # a) a number (4 decimals)
answer_CP4_3b = ...  # b) an integer
answer_CP4_3c = ...  # c) a list (4 decimals)
answer_CP4_3d = ...  # d) a number (3 decimals)
answer_CP4_3e = ...  # e) a list (4 decimals)
answer_CP4_3f = ...  # f) a number (4 decimals)
answer_CP4_3g = ...  # g) a number (4 decimals)
answer_CP4_3h = ...  # h) a list of 2 integers

if exam_over():
    for letter, answer in zip("abcdefgh", [answer_CP4_3a, answer_CP4_3b, answer_CP4_3c, answer_CP4_3d, answer_CP4_3e, answer_CP4_3f, answer_CP4_3g, answer_CP4_3h]):
        RESULTS[f"CP4.3{letter}"] = wbm.check(f"CP4.3{letter}", answer)

**CP4.4 — Gradients par rapport à une matrice**

In [ ]:
# CP4.4: the values written on your answer sheet
answer_CP4_4a = ...  # a) [[a, b], [c, d]]
answer_CP4_4b = ...  # b) [[a, b], [c, d]]
answer_CP4_4c = ...  # c) [[a, b], [c, d]]
answer_CP4_4d = ...  # d) [[a, b], [c, d]]
answer_CP4_4e = ...  # e) a number
answer_CP4_4f = ...  # f) [[a, b], [c, d]]
answer_CP4_4g = ...  # g) [[a, b], [c, d]]

if exam_over():
    for letter, answer in zip("abcdefg", [answer_CP4_4a, answer_CP4_4b, answer_CP4_4c, answer_CP4_4d, answer_CP4_4e, answer_CP4_4f, answer_CP4_4g]):
        RESULTS[f"CP4.4{letter}"] = wbm.check(f"CP4.4{letter}", answer)

**CP4.5 — Hessienne, Taylor d'ordre 2 et nature d'un point critique**

In [ ]:
# CP4.5: the values written on your answer sheet
answer_CP4_5a = ...  # a) a list of 3 numbers
answer_CP4_5b = ...  # b) a list of 3 lists
answer_CP4_5c = ...  # c) a number
answer_CP4_5d = ...  # d) a number
answer_CP4_5e = ...  # e) a list of points [x, y, z]
answer_CP4_5f = ...  # f) a list of 3 numbers
answer_CP4_5g = ...  # g) an integer: 1, 2, 3 or 4
answer_CP4_5h = ...  # h) an integer: 1, 2, 3 or 4

if exam_over():
    for letter, answer in zip("abcdefgh", [answer_CP4_5a, answer_CP4_5b, answer_CP4_5c, answer_CP4_5d, answer_CP4_5e, answer_CP4_5f, answer_CP4_5g, answer_CP4_5h]):
        RESULTS[f"CP4.5{letter}"] = wbm.check(f"CP4.5{letter}", answer)

**CP4.6 — Itération de Richardson et récurrence d'ordre 2**

In [ ]:
# CP4.6: the values written on your answer sheet
answer_CP4_6a = ...  # a) a list of 2 numbers
answer_CP4_6b = ...  # b) a list of 2 numbers
answer_CP4_6c = ...  # c) an irreducible fraction, as text (like "3/8")
answer_CP4_6d = ...  # d) a number
answer_CP4_6e = ...  # e) an integer
answer_CP4_6f = ...  # f) a list of 2 numbers
answer_CP4_6g = ...  # g) a number (4 decimals)

if exam_over():
    for letter, answer in zip("abcdefg", [answer_CP4_6a, answer_CP4_6b, answer_CP4_6c, answer_CP4_6d, answer_CP4_6e, answer_CP4_6f, answer_CP4_6g]):
        RESULTS[f"CP4.6{letter}"] = wbm.check(f"CP4.6{letter}", answer)

**CP4.7 — Moments le long d'axes, einsum et estimateur de Horvitz-Thompson**

In [ ]:
# CP4.7: the values written on your answer sheet
answer_CP4_7a = ...  # a) a shape, as text (like "(4, 5)")
answer_CP4_7b = ...  # b) a list of 2 numbers
answer_CP4_7c = ...  # c) a list of 2 lists of 3 numbers
answer_CP4_7d = ...  # d) a list of 2 lists of 2 numbers
answer_CP4_7e = ...  # e) a number
answer_CP4_7f = ...  # f) a number

if exam_over():
    for letter, answer in zip("abcdef", [answer_CP4_7a, answer_CP4_7b, answer_CP4_7c, answer_CP4_7d, answer_CP4_7e, answer_CP4_7f]):
        RESULTS[f"CP4.7{letter}"] = wbm.check(f"CP4.7{letter}", answer)

**CP4.9 — Corriger une copie : cinq erreurs (les lettres seulement)**

In [ ]:
# CP4.9: the values written on your answer sheet
answer_CP4_9a = ...  # a) the letters, as text (like "ABCDE")

if exam_over():
    for letter, answer in zip("a", [answer_CP4_9a]):
        RESULTS[f"CP4.9{letter}"] = wbm.check(f"CP4.9{letter}", answer)

**CP4.10 — Problème de synthèse : ajuster une loi exponentielle, du graphe de calcul au pas critique**

In [ ]:
# CP4.10: the values written on your answer sheet
answer_CP4_10a = ...  # a) a number
answer_CP4_10b = ...  # b) a list of 3 numbers
answer_CP4_10c = ...  # c) a list of 2 numbers
answer_CP4_10d = ...  # d) a list of 2 numbers
answer_CP4_10e = ...  # e) a number (3 decimals)
answer_CP4_10f = ...  # f) a list (3 decimals)
answer_CP4_10g = ...  # g) [[a, b], [c, d]]
answer_CP4_10h = ...  # h) a list (3 decimals)
answer_CP4_10i = ...  # i) a number (4 decimals)

if exam_over():
    for letter, answer in zip("abcdefghi", [answer_CP4_10a, answer_CP4_10b, answer_CP4_10c, answer_CP4_10d, answer_CP4_10e, answer_CP4_10f, answer_CP4_10g, answer_CP4_10h, answer_CP4_10i]):
        RESULTS[f"CP4.10{letter}"] = wbm.check(f"CP4.10{letter}", answer)

## Bilan

In [ ]:
# Summary of the automatic checks
import re

EXPECTED = ['CP4.1a', 'CP4.1b', 'CP4.1c', 'CP4.1d', 'CP4.1e', 'CP4.1f', 'CP4.1g', 'CP4.1h', 'CP4.2a', 'CP4.2b', 'CP4.2c', 'CP4.2d', 'CP4.2e', 'CP4.2f', 'CP4.2g', 'CP4.2h', 'CP4.3a', 'CP4.3b', 'CP4.3c', 'CP4.3d', 'CP4.3e', 'CP4.3f', 'CP4.3g', 'CP4.3h', 'CP4.4a', 'CP4.4b', 'CP4.4c', 'CP4.4d', 'CP4.4e', 'CP4.4f', 'CP4.4g', 'CP4.5a', 'CP4.5b', 'CP4.5c', 'CP4.5d', 'CP4.5e', 'CP4.5f', 'CP4.5g', 'CP4.5h', 'CP4.6a', 'CP4.6b', 'CP4.6c', 'CP4.6d', 'CP4.6e', 'CP4.6f', 'CP4.6g', 'CP4.7a', 'CP4.7b', 'CP4.7c', 'CP4.7d', 'CP4.7e', 'CP4.7f', 'CP4.9a', 'CP4.10a', 'CP4.10b', 'CP4.10c', 'CP4.10d', 'CP4.10e', 'CP4.10f', 'CP4.10g', 'CP4.10h', 'CP4.10i']


def natural(sub_id):
    """CP4.2a before CP4.10a: the question number as a number, then the letter."""
    number, letter = re.match(r"CP4\.(\d+)(.*)", sub_id).groups()
    return int(number), letter


if exam_over():
    done = {k: r for k, r in RESULTS.items() if r.status != "pending"}
    right = sorted((k for k, r in done.items() if r), key=natural)
    wrong = sorted((k for k, r in done.items() if not r), key=natural)
    pending = sorted((k for k, r in RESULTS.items() if r.status == "pending"), key=natural)
    missing = [k for k in EXPECTED if k not in RESULTS]
    print(f"Réponses vérifiées : {len(right)} juste(s) sur {len(done)} ; pas remplies : {len(pending)}.")
    if wrong:
        print("À revoir avec le corrigé :", ", ".join(wrong))
    if missing:
        print("⚠️ Pas vérifiées (cellule pas exécutée, ou arrêtée par une erreur, par exemple une virgule décimale) :",
              ", ".join(missing))
    print("Note ensuite ta copie avec le barème de 03_examen_corrige.md : il donne aussi les points des démarches, "
          "des justifications et des questions rédigées, que cette vérification ne voit pas.")

**Ce que cette vérification ne voit pas** : les démarches et les justifications ; le quiz CP4.Q1 ; la démonstration CP4.8 ; les corrections de CP4.9 b) (seules les lettres de a) sont vérifiées). Tout cela se corrige avec `03_examen_corrige.md`. Remplis ensuite le tableau de ta note, à la fin de ta copie, puis fais la remédiation du corrigé.